# MIMIC-III NLP Tutorial: spaCy vs scispaCy vs medspaCy
### Data Cohort: Subarachnoid Hemorrhage (ICD-9 430) — Discharge Summaries

This notebook demonstrates entity extraction, word2vec embeddings, and tuned t-SNE
visualizations using three NLP libraries applied to real MIMIC-III clinical notes:

1. **spaCy** — general-purpose NLP
2. **scispaCy** — biomedical-domain NLP
3. **medspaCy** — clinical NLP with context/negation detection



## 1. Setup & Installs

In [3]:
# ==============================================================================
# SECTION 1: ENVIRONMENT SETUP & DEPENDENCY RESOLUTION
# ==============================================================================
!pip install -q --upgrade pip
# Force install compatible data-science stack components
!pip install -q "numpy>=1.24,<2.0" "pandas>=2.0" "gensim>=4.4.0" "scikit-learn>=1.3" matplotlib seaborn

# Install spaCy 3.7.5 to perfectly coordinate with scispaCy 0.5.4 and medspaCy 1.3.x
!pip install -q "spacy==3.7.5" --force-reinstall
!pip install -q scispacy==0.5.4

# Fetch pre-compiled domain-specific biomedical models
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# Install medspaCy and transformers stack
!pip install -q medspaCy==1.3.1 transformers datasets accelerate

# Download general-purpose spaCy architecture small English engine
!python -m spacy download en_core_web_sm -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pyrush 1.0.13 requires spacy>=3.8; python_version >= "3.12", but you have spacy 3.7.5 which is incompatible.
medspacy 1.3.1 requires spacy<4.0,>=3.8; python_version >= "3.12", but you have spacy 3.7.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pyiceberg 0.11.1 requires rich<15.0.0,>=10.11.0, but you have rich 15.0.0 which is incompatible.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompati

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

import spacy
import scispacy
import medspacy
from medspacy.visualization import visualize_ent
from spacy import displacy

import gensim
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
import torch
from transformers import AutoTokenizer, AutoModel

# Establish structural visualization aesthetics
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 8)
np.random.seed(42)
torch.manual_seed(42)

print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")
print(f"spaCy: {spacy.__version__} | medspaCy: {medspacy.__version__}")

Pandas: 2.2.2 | NumPy: 1.26.4
spaCy: 3.8.14 | medspaCy: 1.3.1


## 2. Cohort Extraction (BigQuery)

Pull **discharge summaries** for patients diagnosed with **ICD-9 code 430**



In [ ]:
# ==============================================================================
# SECTION 2: CLIENT AUTHENTICATION AND DYNAMIC COHORT EXTRACTION
# ==============================================================================
from google.colab import auth
from google.cloud import bigquery

# Trigger IAM Authentication overlay
auth.authenticate_user()

# Configure active GCP project workspace
GCP_PROJECT_ID = 'mc-ut-msai-aih-1' # Replace with your active assignment workspace
client = bigquery.Client(project=GCP_PROJECT_ID)

# Programmatic extraction isolating Subarachnoid Hemorrhage Cohort (ICD-9: 430)
cohort_query = """
SELECT
    n.row_id,
    n.subject_id,
    n.hadm_id,
    n.category,
    n.text
FROM `physionet-data.mimiciii_notes.noteevents` n
JOIN (
    SELECT DISTINCT subject_id
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    WHERE icd9_code = '430'
) sah ON n.subject_id = sah.subject_id
WHERE n.category = 'Discharge summary'
"""

print("[INFO] Fetching records from Google Cloud BigQuery framework...")
df_notes = client.query(cohort_query).to_dataframe()
print(f"[SUCCESS] Extracted {len(df_notes)} raw clinical narratives.")

# Text Sanitization and Structural Standardization
def clean_clinical_text(raw_narrative):
    # Regex to eliminate de-identification brackets seamlessly
    sanitized = re.sub(r'\[\*\*.*?\*\*\]', '', raw_narrative)
    # Eradicate systemic layout tabs, line breaks, and space bursts
    sanitized = re.sub(r'\s+', ' ', sanitized)
    return sanitized.strip()

df_notes['clean_text'] = df_notes['text'].apply(clean_clinical_text)

# Sample a statistical representative cohort space for processing efficiency
SAMPLE_SIZE = min(100, len(df_notes))
df_sample = df_notes.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"[INFO] Initializing computational workflow on {SAMPLE_SIZE} records.")

---
## 4. spaCy

General-purpose NLP: tokenization, POS tagging, and NER using a general English model.


In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")

docs_spacy = list(nlp_spacy.pipe(df_sample['clean_text'].tolist()))

# Look at entities in the first note
for ent in docs_spacy[0].ents:
    print(ent.text, "->", ent.label_)


### 4a. displaCy visualization of entities in one note

In [ ]:
from spacy import displacy

displacy.render(docs_spacy[0], style="ent", jupyter=True)


### 4b. Entity extraction across the cohort

In [ ]:
spacy_entities = []
for doc in docs_spacy:
    for ent in doc.ents:
        spacy_entities.append({'text': ent.text, 'label': ent.label_})

df_spacy_ents = pd.DataFrame(spacy_entities)
print("Total entities extracted:", len(df_spacy_ents))
df_spacy_ents['label'].value_counts()


### 4c. word2vec on the note corpus

In [ ]:
# Tokenize each note into a list of lowercase alphabetic tokens
def tokenize(doc):
    return [t.text.lower() for t in doc if t.is_alpha and not t.is_stop]

spacy_sentences = [tokenize(doc) for doc in docs_spacy]

w2v_spacy = Word2Vec(
    sentences=spacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_spacy.wv))


### 4d. Tuned t-SNE

We compare several perplexity values (per the Distill.pub guidance) before settling on one.

In [ ]:
def plot_tsne_perplexities(word2vec_model, perplexities, title_prefix, max_words=300):
    words = list(word2vec_model.wv.index_to_key)[:max_words]
    vectors = np.array([word2vec_model.wv[w] for w in words])

    fig, axes = plt.subplots(1, len(perplexities), figsize=(6 * len(perplexities), 6))
    if len(perplexities) == 1:
        axes = [axes]

    for ax, perp in zip(axes, perplexities):
        tsne = TSNE(
            n_components=2,
            perplexity=min(perp, max(5, len(words) - 1)),
            max_iter=2000,
            init='pca',
            random_state=42,
        )
        coords = tsne.fit_transform(vectors)
        ax.scatter(coords[:, 0], coords[:, 1], s=10, alpha=0.6)
        ax.set_title(f"{title_prefix} (perplexity={perp})")

    plt.tight_layout()
    plt.show()

plot_tsne_perplexities(w2v_spacy, perplexities=[5, 15, 30, 50], title_prefix="spaCy word2vec")


---
## 5. scispaCy



In [ ]:
import spacy

# Only override components that house the embedding architecture directly
config_overrides = {
    "components": {
        "tok2vec": {
            "model": {"embed": {"include_static_vectors": False}}
        },
        "ner": {
            "model": {"tok2vec": {"embed": {"include_static_vectors": False}}}
        }
    }
}

# Load the model with the corrected overrides
nlp_scispacy = spacy.load("en_core_sci_sm", config=config_overrides)

# Process your dataframe as before
docs_scispacy = list(nlp_scispacy.pipe(df_sample['clean_text'].tolist()))

for ent in docs_scispacy[0].ents:
    print(ent.text, "->", ent.label_)

### 5a. Entity extraction across the cohort

In [ ]:
scispacy_entities = []
for doc in docs_scispacy:
    for ent in doc.ents:
        scispacy_entities.append({'text': ent.text, 'label': ent.label_})

df_scispacy_ents = pd.DataFrame(scispacy_entities)
print("Total entities extracted:", len(df_scispacy_ents))
df_scispacy_ents['label'].value_counts()


### 5b. word2vec on the note corpus

In [ ]:
scispacy_sentences = [tokenize(doc) for doc in docs_scispacy]

w2v_scispacy = Word2Vec(
    sentences=scispacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_scispacy.wv))


### 5c. Tuned t-SNE

In [ ]:
plot_tsne_perplexities(w2v_scispacy, perplexities=[5, 15, 30, 50], title_prefix="scispaCy word2vec")


---
## 6. medspaCy



In [ ]:
!pip install -q tqdm

In [ ]:
from tqdm.auto import tqdm

# A smaller subsample
MEDSPACY_SAMPLE_SIZE = min(75, len(df_sample))
df_medspacy_sample = df_sample.iloc[:MEDSPACY_SAMPLE_SIZE].reset_index(drop=True)
print(f"Running medspaCy on {len(df_medspacy_sample)} notes "
      f"(out of {len(df_sample)} in the full sample used for spaCy/scispaCy)")

nlp_medspacy = medspacy.load(disable=["parser"])

docs_medspacy = []
for doc in tqdm(
    nlp_medspacy.pipe(df_medspacy_sample['clean_text'].tolist(), batch_size=5),
    total=len(df_medspacy_sample),
    desc="medspaCy processing notes",
):
    docs_medspacy.append(doc)

for ent in docs_medspacy[0].ents:
    print(ent.text, "-> negated:", ent._.is_negated, "| family:", ent._.is_family, "| hypothetical:", ent._.is_hypothetical)


### 6a. Visualize context attributes (negation, family history, etc.)

In [ ]:
visualize_ent(docs_medspacy[0])


### 6b. Entity extraction across the cohort

In [ ]:
medspacy_entities = []
for doc in docs_medspacy:
    for ent in doc.ents:
        medspacy_entities.append({
            'text': ent.text,
            'label': ent.label_,
            'is_negated': ent._.is_negated,
            'is_family': ent._.is_family,
            'is_hypothetical': ent._.is_hypothetical,
        })

df_medspacy_ents = pd.DataFrame(medspacy_entities)
print("Total entities extracted:", len(df_medspacy_ents))
print("\nNegated entities:", df_medspacy_ents['is_negated'].sum())
df_medspacy_ents['label'].value_counts()


### 6c. word2vec on the note corpus

In [ ]:
medspacy_sentences = [tokenize(doc) for doc in docs_medspacy]

w2v_medspacy = Word2Vec(
    sentences=medspacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_medspacy.wv))


### 6d. Tuned t-SNE

In [ ]:
plot_tsne_perplexities(w2v_medspacy, perplexities=[5, 15, 30, 50], title_prefix="medspaCy word2vec")


---
## 7. Compare & Contrast

Summary table of entity counts and unique labels found by each tool.


In [ ]:
comparison = pd.DataFrame({
    'Tool': ['spaCy', 'scispaCy', 'medspaCy'],
    'Total Entities': [len(df_spacy_ents), len(df_scispacy_ents), len(df_medspacy_ents)],
    'Unique Labels': [df_spacy_ents['label'].nunique(), df_scispacy_ents['label'].nunique(), df_medspacy_ents['label'].nunique()],
})

